# Smash Bros Character + Stage Training
**Before running anything:**
1. Runtime → Change runtime type → **T4 GPU**
2. Make sure `chars.zip` and `stages.zip` are in your Google Drive under `ultdata/`
3. Run cells top to bottom

In [ ]:
# Cell 1 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Mount Drive, unzip, separate chars and stages
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/content/data', exist_ok=True)

!unzip -q /content/drive/MyDrive/ultdata/chars.zip -d /content/data/
!unzip -q /content/drive/MyDrive/ultdata/stages.zip -d /content/data/

os.makedirs('/content/data/chars', exist_ok=True)
os.makedirs('/content/data/stages', exist_ok=True)

for folder in os.listdir('/content/data'):
    fp = os.path.join('/content/data', folder)
    if not os.path.isdir(fp):
        continue
    files = os.listdir(fp)
    if any(f.endswith('.png') for f in files):
        shutil.move(fp, '/content/data/chars/')
    elif any(f.endswith('.jpg') for f in files):
        shutil.move(fp, '/content/data/stages/')

print('chars:', len(os.listdir('/content/data/chars')))
print('stages:', len(os.listdir('/content/data/stages')))

In [ ]:
# Cell 3 — Config
import os
CHARACTER_PNG_DIR = '/content/data/chars'
CONFIDENCE_THRESHOLD = 0.1
STAGE_CONFIDENCE_THRESHOLD = 0.7
PORTRAIT_MASK_CENTER = (63.5, 67)
PORTRAIT_MASK_SIZE = (85, 85)
PORTRAIT_MASK_ANGLE = -27

In [ ]:
# Cell 4 — Train characters
import os, random, re
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

CHARS_DIR = CHARACTER_PNG_DIR
MODEL_PATH = '/content/data/character_model.pth'
CLASSES_PATH = '/content/data/character_classes.npy'

_PNG_RE = re.compile(r'^chara_0_.+_0[0-7]\.png$', re.IGNORECASE)

_KOOPAJR_LABELS = {
    0: 'Bowser Jr',         1: 'Bowser Jr (Larry)',
    2: 'Bowser Jr (Roy)',   3: 'Bowser Jr (Wendy)',
    4: 'Bowser Jr (Iggy)',  5: 'Bowser Jr (Morton)',
    6: 'Bowser Jr (Lemmy)', 7: 'Bowser Jr (Ludwig)',
}
_HERO_LABELS = {
    0: 'Hero (Luminary)', 1: 'Hero (Erdrick)',
    2: 'Hero (Solo)',     3: 'Hero (Eight)',
    4: 'Hero (Luminary)', 5: 'Hero (Erdrick)',
    6: 'Hero (Solo)',     7: 'Hero (Eight)',
}

_train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.5, hue=0.05),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
_val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def _folder_to_name(folder):
    for sep in (' - ', ' . '):
        parts = folder.split(sep, 1)
        if len(parts) == 2:
            return parts[1].strip()
    return folder.strip()

def _file_label(folder_name, filename):
    base = filename.rsplit('.', 1)[0]
    parts = base.split('_')
    alt = int(parts[-1])
    char_id = '_'.join(parts[2:-1])
    if char_id == 'koopajr': return _KOOPAJR_LABELS[alt]
    if char_id == 'brave':   return _HERO_LABELS[alt]
    return _folder_to_name(folder_name)

def _load_img(path):
    buf = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    if img is None: return None
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.ndim == 3 and img.shape[2] == 4:
        bgr = img[:,:,:3].copy()
        bgr[img[:,:,3] == 0] = 0
        return bgr
    return img[:,:,:3]

def _preprocess(bgr):
    resized = cv2.resize(bgr, (128,128), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

class DS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        rgb, label = self.items[i]
        return self.tf(rgb), label

all_labels, folder_files = set(), []
for folder in os.listdir(CHARS_DIR):
    fp = os.path.join(CHARS_DIR, folder)
    if not os.path.isdir(fp): continue
    for fname in os.listdir(fp):
        if not _PNG_RE.match(fname): continue
        label = _file_label(folder, fname)
        all_labels.add(label)
        folder_files.append((fp, fname, label))

classes = sorted(all_labels)
class_to_idx = {c:i for i,c in enumerate(classes)}
items = []
for fp, fname, label in folder_files:
    img = _load_img(os.path.join(fp, fname))
    if img is not None:
        items.append((_preprocess(img), class_to_idx[label]))

np.save(CLASSES_PATH, np.array(classes))
print(f'{len(items)} images, {len(classes)} classes')

random.shuffle(items)
val_size = max(len(classes), len(items)//8)
train_ds = DS(items[val_size:], _train_tf)
val_ds   = DS(items[:val_size], _val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_acc = 0.0
for epoch in range(50):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct/total if total else 0
    marker = ' <-- saved' if val_acc >= best_val_acc else ''
    print(f'Epoch {epoch+1:2d}/50  loss={total_loss/len(train_loader):.3f}  val_acc={val_acc:.1%}{marker}')
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

print(f'Done. Best val accuracy: {best_val_acc:.1%}')

In [ ]:
# Cell 5 — Train stages (all variants labeled as same stage)
STAGES_DIR = '/content/data/stages'
STAGE_MODEL_PATH = '/content/data/stage_model.pth'
STAGE_CLASSES_PATH = '/content/data/stage_classes.npy'

_stage_train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
_stage_val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def _stage_folder_to_name(folder):
    parts = folder.split(' - ', 1)
    return parts[1].strip() if len(parts) == 2 else folder.strip()

def _load_stage_img(path):
    buf = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if img is None: return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

stage_classes = sorted({
    _stage_folder_to_name(f)
    for f in os.listdir(STAGES_DIR)
    if os.path.isdir(os.path.join(STAGES_DIR, f))
})
stage_class_to_idx = {c:i for i,c in enumerate(stage_classes)}

stage_items = []
for folder in os.listdir(STAGES_DIR):
    fp = os.path.join(STAGES_DIR, folder)
    if not os.path.isdir(fp): continue
    name = _stage_folder_to_name(folder)
    for fname in os.listdir(fp):
        if not fname.lower().endswith('.jpg'): continue
        img = _load_stage_img(os.path.join(fp, fname))
        if img is not None:
            stage_items.append((img, stage_class_to_idx[name]))

np.save(STAGE_CLASSES_PATH, np.array(stage_classes))
print(f'{len(stage_items)} images, {len(stage_classes)} stages')

random.shuffle(stage_items)
val_size = max(len(stage_classes), len(stage_items)//8)
stage_train_ds = DS(stage_items[val_size:], _stage_train_tf)
stage_val_ds   = DS(stage_items[:val_size], _stage_val_tf)
stage_train_loader = DataLoader(stage_train_ds, batch_size=32, shuffle=True)
stage_val_loader   = DataLoader(stage_val_ds,   batch_size=32)

stage_model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
stage_model.classifier[3] = nn.Linear(stage_model.classifier[3].in_features, len(stage_classes))
stage_model = stage_model.to(device)

stage_optimizer = torch.optim.Adam(stage_model.parameters(), lr=1e-4)
stage_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(stage_optimizer, T_max=50)

best_stage_acc = 0.0
for epoch in range(50):
    stage_model.train()
    total_loss = 0.0
    for imgs, labels in stage_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        stage_optimizer.zero_grad()
        loss = criterion(stage_model(imgs), labels)
        loss.backward()
        stage_optimizer.step()
        total_loss += loss.item()
    stage_scheduler.step()
    stage_model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in stage_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (stage_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct/total if total else 0
    marker = ' <-- saved' if val_acc >= best_stage_acc else ''
    print(f'Epoch {epoch+1:2d}/50  loss={total_loss/len(stage_train_loader):.3f}  val_acc={val_acc:.1%}{marker}')
    if val_acc >= best_stage_acc:
        best_stage_acc = val_acc
        torch.save(stage_model.state_dict(), STAGE_MODEL_PATH)

print(f'Done. Best val accuracy: {best_stage_acc:.1%}')

In [ ]:
# Cell 6 — Save output files to Google Drive
import shutil
shutil.copy('/content/data/character_model.pth', '/content/drive/MyDrive/ultdata/character_model.pth')
shutil.copy('/content/data/character_classes.npy', '/content/drive/MyDrive/ultdata/character_classes.npy')
shutil.copy('/content/data/stage_model.pth', '/content/drive/MyDrive/ultdata/stage_model.pth')
shutil.copy('/content/data/stage_classes.npy', '/content/drive/MyDrive/ultdata/stage_classes.npy')
print('All 4 files saved to Google Drive ultdata folder')